# ATIS Tag Distribution Analysis

Goal: understand the distribution of **intents** and **slot tags** in the ATIS dataset so we can
decide which fields to include in our curated JSON output schema for the structured-output
fine-tuning experiment.

Data: `data/atis_train.csv` and `data/atis_test.csv` (columns: `id`, `intent`, `text`, `slots`).
The `slots` column holds space-separated BIO tags aligned to the whitespace tokens in `text`.

In [2]:
import pandas as pd
from collections import Counter

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/atis_train.csv")
test = pd.read_csv(f"{DATA_DIR}/atis_test.csv")

print("train:", train.shape, "| test:", test.shape)
train.head()

train: (4978, 4) | test: (893, 4)


,id,intent,text,slots
0,0,flight,i want to fly from boston at 838 am and arrive...,O O O O O B-fromloc.city_name O B-depart_time....
1,1,flight,what flights are available from pittsburgh to ...,O O O O O B-fromloc.city_name O B-toloc.city_n...
2,2,flight_time,what is the arrival time in san francisco for ...,O O O B-flight_time I-flight_time O B-fromloc....
3,3,airfare,cheapest airfare from tacoma to orlando,B-cost_relative O O B-fromloc.city_name O B-to...
4,4,airfare,round trip fares from pittsburgh to philadelph...,B-round_trip I-round_trip O O B-fromloc.city_n...


In [3]:
# Work on the combined set for distribution analysis (train + test).
df = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)
print("combined:", df.shape)

# Sanity check: token count must equal tag count for every row.
tok_len = df["text"].str.split().str.len()
tag_len = df["slots"].str.split().str.len()
mismatch = (tok_len != tag_len).sum()
print("rows where #tokens != #tags:", mismatch)

combined: (5871, 5)
rows where #tokens != #tags: 0


## 1. Intent distribution

ATIS has ~18 intent labels (some rare ones are compound, e.g. `flight+airfare`).

In [4]:
intent_counts = df["intent"].value_counts()
intent_dist = pd.DataFrame({
    "count": intent_counts,
    "pct": (intent_counts / len(df) * 100).round(2),
})
print(f"distinct intents: {df['intent'].nunique()}")
intent_dist

distinct intents: 26


,count,pct
intent,,
flight,4298,73.21
airfare,471,8.02
ground_service,291,4.96
airline,195,3.32
abbreviation,180,3.07
aircraft,90,1.53
flight_time,55,0.94
quantity,54,0.92
airport,38,0.65


## 2. Slot tag parsing

Explode the BIO tags into a long form so we can count them. We track three views:
- **raw tag** (e.g. `B-fromloc.city_name`, `I-fromloc.city_name`, `O`)
- **base slot type** (BIO prefix stripped, e.g. `fromloc.city_name`)
- **slot family** (top level before the dot, e.g. `fromloc`)

In [5]:
# One row per (example, tag) token.
tags_long = df[["id", "split", "slots"]].copy()
tags_long["tag"] = tags_long["slots"].str.split()
tags_long = tags_long.explode("tag")

# Derived columns.
tags_long["is_entity"] = tags_long["tag"] != "O"
tags_long["bio"] = tags_long["tag"].str[:1].where(tags_long["is_entity"], "O")
tags_long["base"] = tags_long["tag"].str.replace(r"^[BI]-", "", regex=True)
tags_long.loc[~tags_long["is_entity"], "base"] = pd.NA
tags_long["family"] = tags_long["base"].str.split(".").str[0]

print("total tokens:", len(tags_long))
print("entity tokens:", int(tags_long['is_entity'].sum()),
      f"({tags_long['is_entity'].mean()*100:.1f}% of tokens; the rest are 'O')")
tags_long.head()

total tokens: 65789
entity tokens: 24210 (36.8% of tokens; the rest are 'O')


,id,split,slots,tag,is_entity,bio,base,family
0,0,train,O O O O O B-fromloc.city_name O B-depart_time....,O,False,O,NaN,NaN
0,0,train,O O O O O B-fromloc.city_name O B-depart_time....,O,False,O,NaN,NaN
0,0,train,O O O O O B-fromloc.city_name O B-depart_time....,O,False,O,NaN,NaN
0,0,train,O O O O O B-fromloc.city_name O B-depart_time....,O,False,O,NaN,NaN
0,0,train,O O O O O B-fromloc.city_name O B-depart_time....,O,False,O,NaN,NaN


## 3. Raw tag frequency (excluding `O`)

In [6]:
raw = tags_long[tags_long["is_entity"]]["tag"].value_counts()
print(f"distinct non-O tags: {raw.size}")
raw.head(40)

distinct non-O tags: 128


tag
B-toloc.city_name               5059
B-fromloc.city_name             5030
I-toloc.city_name               1366
B-depart_date.day_name          1101
I-fromloc.city_name              871
B-airline_name                   802
B-depart_time.period_of_day      723
I-airline_name                   491
B-depart_date.day_number         450
B-depart_date.month_name         435
B-depart_time.time               426
B-round_trip                     421
I-round_trip                     410
B-depart_time.time_relative      388
B-cost_relative                  381
B-flight_mod                     353
I-depart_time.time               351
B-city_name                      284
B-stoploc.city_name              259
B-arrive_time.time               242
B-class_type                     241
B-arrive_time.time_relative      218
I-class_type                     200
I-arrive_time.time               196
B-flight_stop                    189
B-airline_code                   170
I-depart_date.day_number         1

## 4. Entity-level frequency per slot type

Counting **`B-` tags** gives the number of distinct entity *mentions* per slot type — this is the
most relevant signal for picking schema fields (how often each field actually carries a value).
We also show **document frequency**: in how many queries the slot appears at least once.

In [7]:
begins = tags_long[tags_long["bio"] == "B"]

mentions = begins["base"].value_counts().rename("mentions")
doc_freq = begins.groupby("base")["id"].nunique().rename("doc_freq")

slot_stats = pd.concat([mentions, doc_freq], axis=1)
slot_stats["doc_pct"] = (slot_stats["doc_freq"] / len(df) * 100).round(2)
slot_stats = slot_stats.sort_values("mentions", ascending=False)
print(f"distinct slot types: {len(slot_stats)}")
slot_stats.head(40)

distinct slot types: 83


,mentions,doc_freq,doc_pct
base,,,
toloc.city_name,5059,4287,73.02
fromloc.city_name,5030,4296,73.17
depart_date.day_name,1101,1010,17.20
airline_name,802,768,13.08
depart_time.period_of_day,723,656,11.17
depart_date.day_number,450,438,7.46
depart_date.month_name,435,428,7.29
depart_time.time,426,413,7.03
round_trip,421,406,6.92


## 5. Slot families (grouped)

Many fine-grained slots collapse naturally into families (`fromloc.city_name`,
`fromloc.airport_name`, `fromloc.state_code` -> `fromloc`). These families are good candidates
for the curated JSON schema fields.

In [8]:
fam_mentions = begins["family"].value_counts().rename("mentions")
fam_doc = begins.groupby("family")["id"].nunique().rename("doc_freq")
fam_stats = pd.concat([fam_mentions, fam_doc], axis=1)
fam_stats["doc_pct"] = (fam_stats["doc_freq"] / len(df) * 100).round(2)
fam_stats["n_subtypes"] = begins.groupby("family")["base"].nunique()
fam_stats = fam_stats.sort_values("doc_freq", ascending=False)
fam_stats

,mentions,doc_freq,doc_pct,n_subtypes
family,,,,
fromloc,5276,4389,74.76,5
toloc,5338,4340,73.92,6
depart_date,2206,1460,24.87,6
depart_time,1642,1028,17.51,6
airline_name,802,768,13.08,1
round_trip,421,406,6.92,1
cost_relative,381,372,6.34,1
flight_mod,353,344,5.86,1
arrive_time,591,306,5.21,6


## 6. Slots per query

How many entity mentions does a typical query carry? This tells us how sparse the output will be
(i.e. how many fields are `null` per example under a fixed schema).

In [9]:
slots_per_query = begins.groupby("id").size()
# Reindex to include queries with zero entities, if any.
slots_per_query = slots_per_query.reindex(df["id"], fill_value=0)
print(slots_per_query.describe().round(2))
print("\ndistribution of #slots per query:")
slots_per_query.value_counts().sort_index()

count    5871.00
mean        4.29
std         2.34
min         0.00
25%         3.00
50%         4.00
75%         6.00
max        18.00
dtype: float64

distribution of #slots per query:


0       16
1      439
2      919
3     1126
4     1083
5      785
6      533
7      412
8      206
9      152
10     129
11      31
12      13
13       8
14      10
15       4
16       2
17       1
18       2
Name: count, dtype: int64

## 7. Takeaways for the JSON schema

Use the tables above (sections 4 and 5) to choose the curated fields. Rule of thumb:
- Include families with high `doc_pct` (appear in a meaningful share of queries).
- Collapse the long tail of rare slot types into the chosen families (or drop them).
- Expect most fields to be `null` per example (section 6 shows the typical query is sparse),
  which is exactly what the fixed-schema-with-nulls design handles cleanly.

In [10]:
train[train["intent"] == "ground_service"]

,id,intent,text,slots
9,9,ground_service,what kind of ground transportation is availabl...,O O O O O O O O B-city_name
16,16,ground_service,show me the ground transportation in denver,O O O O O O B-city_name
28,28,ground_service,atlanta ground transportation,B-city_name O O
46,46,ground_service,show me information on ground transportation f...,O O O O O O O B-city_name
63,63,ground_service,what is the ground transportation from philade...,O O O O O O B-fromloc.airport_name I-fromloc.a...
...,...,...,...,...
4872,4872,ground_service,can i get a taxi from long beach airport to do...,O O O O B-transport_type O B-fromloc.city_name...
4877,4877,ground_service,what ground transportation is available from t...,O O O O O O O B-fromloc.airport_name I-fromloc...
4932,4932,ground_service,what ground transportation is available from t...,O O O O O O O B-fromloc.airport_name I-fromloc...
4940,4940,ground_service,show me ground transportation in denver,O O O O O B-city_name
